# Cronbach's Alpha Reliability Analysis

This notebook computes Cronbach's alpha for the Likert-scale constructs used in the blended learning thesis project.

The analysis uses the final cleaned dataset:

```text
data/processed/cleaned_data.csv
```

The output files are saved to:

```text
data/processed/reliability/
```

The overall 33-item reliability analysis includes all ordinal Likert-scale variables. The item `tech_issues_freq` is reverse-coded before calculating the overall reliability because higher original values indicate more frequent technical issues, while most other Likert-scale items are positively oriented.


## 1. Import libraries and set paths

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# This notebook is designed to be placed inside the notebook/ folder.
DATA_PATH = Path("../data/processed/cleaned_data.csv")
OUTPUT_DIR = Path("../data/processed/reliability")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset path:", DATA_PATH)
print("Output directory:", OUTPUT_DIR)


Dataset path: ..\data\processed\cleaned_data.csv
Output directory: ..\data\processed\reliability


## 2. Define helper functions

In [2]:
def cronbach_alpha(data: pd.DataFrame):
    """
    Compute Cronbach's alpha for Likert-scale items.

    Parameters
    ----------
    data : pd.DataFrame
        DataFrame containing only the items used in one scale or construct.

    Returns
    -------
    alpha : float
        Cronbach's alpha value.
    n_valid : int
        Number of complete valid responses used.
    k_items : int
        Number of items used.
    """

    data = data.apply(pd.to_numeric, errors="coerce")
    data = data.dropna(axis=0, how="any")

    n_valid = len(data)
    k_items = data.shape[1]

    if k_items < 2 or n_valid == 0:
        return np.nan, n_valid, k_items

    item_variances = data.var(axis=0, ddof=1)
    total_score = data.sum(axis=1)
    total_variance = total_score.var(ddof=1)

    if total_variance == 0:
        return np.nan, n_valid, k_items

    alpha = (k_items / (k_items - 1)) * (
        1 - item_variances.sum() / total_variance
    )

    return alpha, n_valid, k_items


def interpret_alpha(alpha: float) -> str:
    """
    Interpret Cronbach's alpha using common academic thresholds.
    """

    if pd.isna(alpha):
        return "Not available"
    elif alpha >= 0.90:
        return "Excellent"
    elif alpha >= 0.80:
        return "Good"
    elif alpha >= 0.70:
        return "Acceptable"
    elif alpha >= 0.60:
        return "Questionable / moderate"
    else:
        return "Low / questionable"


def alpha_if_item_deleted(data: pd.DataFrame) -> pd.DataFrame:
    """
    Compute Cronbach's alpha after deleting each item one by one.
    This helps identify whether one item reduces the reliability of a construct.
    """

    results = []

    for item in data.columns:
        remaining_items = [col for col in data.columns if col != item]
        alpha, n_valid, k_items = cronbach_alpha(data[remaining_items])

        results.append(
            {
                "Deleted Item": item,
                "Remaining Items": k_items,
                "Valid Responses": n_valid,
                "Alpha if Item Deleted": round(alpha, 3)
                if not pd.isna(alpha)
                else np.nan,
            }
        )

    return pd.DataFrame(results)


## 3. Load the cleaned dataset

In [3]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}\n"
        "Please place this notebook in the notebook/ folder, or adjust DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()


Dataset shape: (588, 87)


,gender,age,is_itc_student,itc_campus,province,itc_student_id,education_level,department,faculty,academic_year,...,career_preparation,ideal_balance,prefer_more_blended,open_strengths,open_challenges_suggestions,survey_start,survey_end,response_time_minutes,student_id,flag_speeder
0,Male,36,True,ITC Phnom Penh (Main Campus),Phnom Penh,e20210528,Year5 - Final Year,GIC,NaN,2021–2022,...,2,"More Online than Face-to-Face (e.g., 40% In-pe...",No,NaN,NaN,2026-03-12 00:35:35.641,2026-03-12 00:37:43.468,2.130450,e20210528,True
1,Male,23,True,ITC Phnom Penh (Main Campus),Phnom Penh,e20210686,Year5 - Final Year,GIC,NaN,2025–2026,...,3,"More Online than Face-to-Face (e.g., 40% In-pe...",Neutral/Unsure,Nothing,Nothing,2026-03-17 18:53:40.435,2026-03-17 18:56:16.257,2.597033,e20210686,True
2,Male,19,True,ITC Phnom Penh (Main Campus),Phnom Penh,e20241146,Year2 - Sophomore,Foundation Year,NaN,2024–2025,...,4,"Balanced Half and Half (50% In-person, 50% Onl...",Neutral/Unsure,"Very good, excellent","No big challenge, i’m the best",2026-03-09 15:33:21.321,2026-03-09 15:36:31.387,3.167767,e20241146,False
3,Female,19,True,ITC Phnom Penh (Main Campus),Phnom Penh,e20240609,Year2 - Sophomore,GIC,NaN,2024–2025,...,3,"Mostly Face-to-Face (e.g., 80% In-person, 20% ...",Neutral/Unsure,Will try hard,Lack of self-discipline,2026-03-09 15:32:15.225,2026-03-09 15:35:36.121,3.348267,e20240609,False
4,Female,18,True,ITC Phnom Penh (Main Campus),Phnom Penh,e20240542,Year2 - Sophomore,GIC,NaN,2024–2025,...,3,"Balanced Half and Half (50% In-person, 50% Onl...",Neutral/Unsure,getting more experience,Discipline on daily studying,2026-03-09 15:31:35.190,2026-03-09 15:35:12.175,3.616417,e20240542,False


## 4. Define Likert-scale constructs

The constructs below organize the 33 ordinal Likert-scale variables into eight questionnaire dimensions used for reliability analysis. The variable `tech_issues_freq` is treated as a single technical issues indicator, so it is not included as a separate construct-level Cronbach's alpha scale. However, it is reverse-coded and included in the overall 33-item reliability analysis.

In [4]:
constructs = {
    "Learning Material Use": [
        "use_lecture_slides",
        "use_video_lectures",
        "use_quizzes",
        "use_articles",
        "use_forums",
        "use_simulations",
    ],

    "Engagement and Interaction": [
        "online_discussion_participation",
        "peer_collaboration",
        "comfort_asking_questions",
        "sense_of_community",
    ],

    "Course Integration and Learning Understanding": [
        "integration_quality",
        "overall_understanding",
    ],

    "Lecturer Support": [
        "lect_clear_instructions",
        "lect_responsive",
        "lect_diverse_tools",
        "lect_timely_feedback",
        "lect_foster_interaction",
    ],

    "Self-Regulation": [
        "self_prioritize_deadlines",
        "self_study_schedule",
        "self_prepare_class",
        "self_responsibility",
    ],

    "Perceived Benefits": [
        "benefit_flexibility",
        "benefit_variety",
        "benefit_recorded_access",
        "benefit_self_study_time",
        "benefit_life_balance",
        "benefit_self_directed",
    ],

    "Digital Learning Readiness and Usability": [
        "video_helpfulness",
        "digital_literacy_improvement",
        "lms_usability",
    ],

    "Learning Outcome and Future Readiness": [
        "overall_satisfaction",
        "career_preparation",
    ],
}

all_33_likert_items = [
    "video_helpfulness",
    "digital_literacy_improvement",
    "use_lecture_slides",
    "use_video_lectures",
    "use_quizzes",
    "use_articles",
    "use_forums",
    "use_simulations",
    "online_discussion_participation",
    "peer_collaboration",
    "comfort_asking_questions",
    "sense_of_community",
    "integration_quality",
    "benefit_flexibility",
    "benefit_variety",
    "benefit_recorded_access",
    "benefit_self_study_time",
    "benefit_life_balance",
    "benefit_self_directed",
    "overall_understanding",
    "lect_clear_instructions",
    "lect_responsive",
    "lect_diverse_tools",
    "lect_timely_feedback",
    "lect_foster_interaction",
    "self_prioritize_deadlines",
    "self_study_schedule",
    "self_prepare_class",
    "self_responsibility",
    "overall_satisfaction",
    "career_preparation",
    "tech_issues_freq",
    "lms_usability",
]

print("Number of constructs:", len(constructs))
print("Number of construct-level items:", sum(len(items) for items in constructs.values()))
print("Number of overall Likert items:", len(all_33_likert_items))

Number of constructs: 8
Number of construct-level items: 32
Number of overall Likert items: 33


## 5. Check that all required columns exist

In [5]:
required_columns = set(all_33_likert_items)

for items in constructs.values():
    required_columns.update(items)

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(
        "The following required columns are missing from the cleaned dataset:\n"
        + "\n".join(missing_columns)
    )

print("All required Likert-scale columns are available.")


All required Likert-scale columns are available.


## 6. Reverse-code `tech_issues_freq`

The original `tech_issues_freq` item is negatively oriented:

- Higher original value = more frequent technical issues
- Most other Likert items: higher value = more positive experience

Therefore, it is reverse-coded before being included in the overall 33-item Cronbach's alpha calculation.


In [6]:
df_alpha = df.copy()

df_alpha["tech_issues_freq"] = pd.to_numeric(
    df_alpha["tech_issues_freq"], errors="coerce"
)

df_alpha["tech_issues_freq_reversed"] = 6 - df_alpha["tech_issues_freq"]

all_33_likert_items_for_alpha = [
    "tech_issues_freq_reversed" if col == "tech_issues_freq" else col
    for col in all_33_likert_items
]

df_alpha[["tech_issues_freq", "tech_issues_freq_reversed"]].head()


,tech_issues_freq,tech_issues_freq_reversed
0,2,4
1,3,3
2,3,3
3,4,2
4,3,3


## 7. Compute Cronbach's alpha

In [7]:
results = []

# Overall 33-item reliability
overall_alpha, overall_n, overall_k = cronbach_alpha(
    df_alpha[all_33_likert_items_for_alpha]
)

results.append(
    {
        "Construct": "Overall Likert-Scale Item Pool",
        "Number of Items": overall_k,
        "Valid Responses": overall_n,
        "Cronbach Alpha": round(overall_alpha, 3)
        if not pd.isna(overall_alpha)
        else np.nan,
        "Interpretation": interpret_alpha(overall_alpha),
    }
)

# Construct-level reliability
for construct_name, items in constructs.items():
    construct_data = df[items]

    alpha, n_valid, k_items = cronbach_alpha(construct_data)

    results.append(
        {
            "Construct": construct_name,
            "Number of Items": k_items,
            "Valid Responses": n_valid,
            "Cronbach Alpha": round(alpha, 3)
            if not pd.isna(alpha)
            else np.nan,
            "Interpretation": interpret_alpha(alpha),
        }
    )

results_df = pd.DataFrame(results)
results_df

,Construct,Number of Items,Valid Responses,Cronbach Alpha,Interpretation
0,Overall Likert-Scale Item Pool,33,588,0.881,Good
1,Learning Material Use,6,588,0.624,Questionable / moderate
2,Engagement and Interaction,4,588,0.570,Low / questionable
3,Course Integration and Learning Understanding,2,588,0.468,Low / questionable
4,Lecturer Support,5,588,0.814,Good
5,Self-Regulation,4,588,0.606,Questionable / moderate
6,Perceived Benefits,6,588,0.773,Acceptable
7,Digital Learning Readiness and Usability,3,588,0.498,Low / questionable
8,Learning Outcome and Future Readiness,2,588,0.661,Questionable / moderate


In [8]:
construct_item_count = sum(len(items) for items in constructs.values())

print("Number of constructs:", len(constructs))
print("Number of construct-level items:", construct_item_count)
print("Number of overall Likert items:", len(all_33_likert_items))
print("Items not included in constructs:")

construct_items = set(item for items in constructs.values() for item in items)
overall_items = set(all_33_likert_items)

missing_from_constructs = overall_items - construct_items
print(missing_from_constructs)

Number of constructs: 8
Number of construct-level items: 32
Number of overall Likert items: 33
Items not included in constructs:
{'tech_issues_freq'}


The 33 ordinal Likert-scale variables were grouped into eight multi-item constructs for construct-level reliability analysis. The variable \texttt{tech\_issues\_freq} was treated as a single technical issues indicator and was therefore not assessed as a separate construct using Cronbach's alpha. However, it was reverse-coded and included in the overall 33-item reliability analysis.

## 8. Compute alpha-if-item-deleted diagnostics

## What “Alpha if Item Deleted” Means

The alpha-if-item-deleted analysis is a diagnostic step used to examine whether each questionnaire item contributes positively to the internal consistency of a scale or construct. It recalculates Cronbach's alpha after removing one item at a time.

If the alpha value becomes higher after deleting an item, it may indicate that the item is less consistent with the other items in the same construct. If the alpha value becomes lower or stays almost the same, the item does not appear to reduce the reliability of the construct.

In this study, alpha-if-item-deleted was used only as a diagnostic check. Items were not automatically removed based only on this result because the questionnaire variables were selected based on their relevance to blended learning behavior and perception. Therefore, item removal was considered only when both statistical evidence and theoretical meaning supported it.


### How to Interpret Alpha-if-Item-Deleted

The alpha-if-item-deleted result shows what Cronbach's alpha would become if one item was removed from the item pool or construct. It is used as a diagnostic check to see whether any item reduces internal consistency.

| Case | Meaning |
|---|---|
| Alpha if item deleted is lower than original alpha | Keep the item |
| Alpha if item deleted is much higher than original alpha | The item may not fit well |
| Alpha if item deleted is almost the same | The item is not harmful |

In this study, alpha-if-item-deleted was used only as a diagnostic tool. Items were not removed automatically because each variable was selected based on its relevance to blended learning behavior and perception.

In [9]:
# =====================================================
# ALPHA-IF-ITEM-DELETED DIAGNOSTICS
# =====================================================

# 1. Overall 33-item alpha-if-item-deleted
overall_deleted = alpha_if_item_deleted(
    df_alpha[all_33_likert_items_for_alpha]
)

overall_deleted_path = OUTPUT_DIR / "alpha_if_item_deleted_overall_33_items.csv"

overall_deleted.to_csv(
    overall_deleted_path,
    index=False,
    encoding="utf-8-sig",
)

print(f"Saved overall alpha-if-item-deleted diagnostics: {overall_deleted_path}")


# 2. Construct-level alpha-if-item-deleted
skipped_constructs = []

for construct_name, items in constructs.items():
    item_count = len(items)

    # Alpha-if-item-deleted is not meaningful for constructs with fewer than 3 items.
    # For a 2-item construct, deleting one item leaves only one item.
    if item_count < 3:
        skipped_constructs.append(
            {
                "Construct": construct_name,
                "Number of Items": item_count,
                "Reason": "Skipped because alpha-if-item-deleted is not meaningful for constructs with fewer than 3 items.",
            }
        )
        continue

    construct_data = df[items]

    deleted_result = alpha_if_item_deleted(construct_data)

    safe_name = (
        construct_name.lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("-", "_")
        .replace("&", "and")
    )

    output_path = OUTPUT_DIR / f"alpha_if_item_deleted_{safe_name}.csv"

    deleted_result.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig",
    )

    print(f"Saved construct alpha-if-item-deleted diagnostics: {output_path}")


# 3. Save skipped construct note
if skipped_constructs:
    skipped_df = pd.DataFrame(skipped_constructs)

    skipped_path = OUTPUT_DIR / "alpha_if_item_deleted_skipped_constructs.csv"

    skipped_df.to_csv(
        skipped_path,
        index=False,
        encoding="utf-8-sig",
    )

    print(f"Saved skipped construct note: {skipped_path}")


print("\nAlpha-if-item-deleted diagnostics completed.")

overall_deleted.head()

Saved overall alpha-if-item-deleted diagnostics: ..\data\processed\reliability\alpha_if_item_deleted_overall_33_items.csv
Saved construct alpha-if-item-deleted diagnostics: ..\data\processed\reliability\alpha_if_item_deleted_learning_material_use.csv
Saved construct alpha-if-item-deleted diagnostics: ..\data\processed\reliability\alpha_if_item_deleted_engagement_and_interaction.csv
Saved construct alpha-if-item-deleted diagnostics: ..\data\processed\reliability\alpha_if_item_deleted_lecturer_support.csv
Saved construct alpha-if-item-deleted diagnostics: ..\data\processed\reliability\alpha_if_item_deleted_self_regulation.csv
Saved construct alpha-if-item-deleted diagnostics: ..\data\processed\reliability\alpha_if_item_deleted_perceived_benefits.csv
Saved construct alpha-if-item-deleted diagnostics: ..\data\processed\reliability\alpha_if_item_deleted_digital_learning_readiness_and_usability.csv
Saved skipped construct note: ..\data\processed\reliability\alpha_if_item_deleted_skipped_cons

,Deleted Item,Remaining Items,Valid Responses,Alpha if Item Deleted
0,video_helpfulness,32,588,0.879
1,digital_literacy_improvement,32,588,0.876
2,use_lecture_slides,32,588,0.878
3,use_video_lectures,32,588,0.878
4,use_quizzes,32,588,0.879


Alpha-if-item-deleted diagnostics were computed for the overall 33-item Likert-scale pool and for constructs containing at least three items. Two-item constructs were not included in the item-deletion diagnostics because removing one item would leave only a single item, making Cronbach's alpha no longer meaningful.

## 9. Save reliability results for thesis appendix

In [10]:
results_csv_path = OUTPUT_DIR / "cronbach_alpha_results.csv"
results_tex_path = OUTPUT_DIR / "cronbach_alpha_table.tex"

results_df.to_csv(
    results_csv_path,
    index=False,
    encoding="utf-8-sig",
)

latex_table = results_df.to_latex(
    index=False,
    escape=False,
    caption="Cronbach's Alpha Reliability Results for Likert-Scale Constructs",
    label="tab:cronbach-alpha",
    column_format="lcccl",
    float_format="%.3f",
)

with open(results_tex_path, "w", encoding="utf-8") as f:
    f.write(latex_table)

print("Saved output files:")
print(f"- {results_csv_path}")
print(f"- {results_tex_path}")
print(f"- {OUTPUT_DIR / 'alpha_if_item_deleted_overall_33_items.csv'}")


Saved output files:
- ..\data\processed\reliability\cronbach_alpha_results.csv
- ..\data\processed\reliability\cronbach_alpha_table.tex
- ..\data\processed\reliability\alpha_if_item_deleted_overall_33_items.csv


##  Thesis Interpretation

Use this wording in the methodology section or appendix after confirming the generated reliability table values:


To assess the reliability of the Likert-scale items used in the survey, Cronbach's alpha was computed using the final cleaned dataset. The reliability analysis was conducted after preprocessing, using the 33 ordinal Likert-scale variables that were later used for multivariate analysis and clustering. Since the cleaned dataset contained no missing values in these 33 variables, all 588 valid responses were included in the reliability calculation.

For the overall 33-item reliability analysis, the item \texttt{tech\_issues\_freq} was reverse-coded before computing Cronbach's alpha because higher original values indicated more frequent technical problems, whereas most other Likert-scale items were positively oriented.

The overall Cronbach's alpha for the 33-item Likert-scale pool was \(\alpha = 0.881\), indicating good internal consistency. Construct-level reliability was also examined because the questionnaire covered multiple dimensions of blended learning experience. Among the construct-level results, lecturer support showed good reliability with \(\alpha = 0.814\), while perceived benefits showed acceptable reliability with \(\alpha = 0.773\). Learning material use, self-regulation, and learning outcome and future readiness showed moderate reliability values. Engagement and interaction, course integration and learning understanding, and digital learning readiness and usability showed lower reliability values and were therefore interpreted cautiously.

The alpha-if-item-deleted diagnostics did not indicate a strong need to remove any item. For the overall 33-item pool, deleting \texttt{tech\_issues\_freq\_reversed} slightly increased the alpha value from 0.881 to 0.886; however, the increase was very small, and the item was retained because technical issues are theoretically important in the context of blended learning. Two-item constructs were not interpreted using alpha-if-item-deleted diagnostics because deleting one item would leave only a single item, making Cronbach's alpha no longer meaningful.

Overall, the reliability results support the use of the 33 ordinal Likert-scale variables for exploratory analysis, clustering, and student profile interpretation. However, because some construct-level alpha values were low and the questionnaire was designed for this study, the instrument should be treated as an exploratory perception survey rather than a fully validated psychometric scale.



In [12]:
# =====================================================
# INTERPRET ALPHA-IF-ITEM-DELETED RESULTS
# =====================================================

alpha_deleted_files = {
    "Overall Likert-Scale Item Pool": "alpha_if_item_deleted_overall_33_items.csv",
    "Learning Material Use": "alpha_if_item_deleted_learning_material_use.csv",
    "Engagement and Interaction": "alpha_if_item_deleted_engagement_and_interaction.csv",
    "Course Integration and Learning Understanding": "alpha_if_item_deleted_course_integration_and_learning_understanding.csv",
    "Lecturer Support": "alpha_if_item_deleted_lecturer_support.csv",
    "Self-Regulation": "alpha_if_item_deleted_self_regulation.csv",
    "Perceived Benefits": "alpha_if_item_deleted_perceived_benefits.csv",
    "Digital Learning Readiness and Usability": "alpha_if_item_deleted_digital_learning_readiness_and_usability.csv",
    "Learning Outcome and Future Readiness": "alpha_if_item_deleted_learning_outcome_and_future_readiness.csv",
}

interpretation_results = []

for _, row in results_df.iterrows():
    construct_name = row["Construct"]
    original_alpha = row["Cronbach Alpha"]

    file_name = alpha_deleted_files.get(construct_name)

    if file_name is None:
        continue

    file_path = OUTPUT_DIR / file_name

    if not file_path.exists():
        interpretation_results.append(
            {
                "Construct": construct_name,
                "Original Alpha": original_alpha,
                "Highest Alpha if Item Deleted": None,
                "Item with Highest Deleted Alpha": None,
                "Difference": None,
                "Decision": "No alpha-if-item-deleted file found.",
            }
        )
        continue

    deleted_df = pd.read_csv(file_path)

    if deleted_df["Alpha if Item Deleted"].isna().all():
        interpretation_results.append(
            {
                "Construct": construct_name,
                "Original Alpha": original_alpha,
                "Highest Alpha if Item Deleted": None,
                "Item with Highest Deleted Alpha": None,
                "Difference": None,
                "Decision": "Skip interpretation because this is a 2-item construct. Deleting one item leaves only one item.",
            }
        )
        continue

    max_idx = deleted_df["Alpha if Item Deleted"].idxmax()
    highest_alpha = deleted_df.loc[max_idx, "Alpha if Item Deleted"]
    item_name = deleted_df.loc[max_idx, "Deleted Item"]
    difference = highest_alpha - original_alpha

    if difference > 0.020:
        decision = "Review this item because deleting it clearly improves alpha."
    elif difference > 0.005:
        decision = "Small increase only. Keep the item unless there is a strong theoretical reason to remove it."
    elif difference > 0:
        decision = "Very small increase. Keep the item."
    else:
        decision = "Keep all items because deleting any item does not improve alpha."

    interpretation_results.append(
        {
            "Construct": construct_name,
            "Original Alpha": original_alpha,
            "Highest Alpha if Item Deleted": highest_alpha,
            "Item with Highest Deleted Alpha": item_name,
            "Difference": round(difference, 3),
            "Decision": decision,
        }
    )

alpha_deleted_interpretation_df = pd.DataFrame(interpretation_results)
alpha_deleted_interpretation_df

,Construct,Original Alpha,Highest Alpha if Item Deleted,Item with Highest Deleted Alpha,Difference,Decision
0,Overall Likert-Scale Item Pool,0.881,0.886,tech_issues_freq_reversed,0.005,Small increase only. Keep the item unless ther...
1,Learning Material Use,0.624,0.605,use_lecture_slides,-0.019,Keep all items because deleting any item does ...
2,Engagement and Interaction,0.570,0.511,comfort_asking_questions,-0.059,Keep all items because deleting any item does ...
3,Course Integration and Learning Understanding,0.468,NaN,NaN,NaN,Skip interpretation because this is a 2-item c...
4,Lecturer Support,0.814,0.806,lect_timely_feedback,-0.008,Keep all items because deleting any item does ...
5,Self-Regulation,0.606,0.572,self_prioritize_deadlines,-0.034,Keep all items because deleting any item does ...
6,Perceived Benefits,0.773,0.754,benefit_recorded_access,-0.019,Keep all items because deleting any item does ...
7,Digital Learning Readiness and Usability,0.498,0.464,lms_usability,-0.034,Keep all items because deleting any item does ...
8,Learning Outcome and Future Readiness,0.661,NaN,NaN,NaN,Skip interpretation because this is a 2-item c...


The alpha-if-item-deleted results were compared with the original Cronbach's alpha of each construct. In most constructs, deleting any item produced a lower alpha value, indicating that the items should be retained. For the overall 33-item Likert-scale pool, deleting `tech_issues_freq_reversed` increased alpha slightly from 0.881 to 0.886. However, the improvement was very small, and the item was retained because technical issues are theoretically important in blended learning. Two-item constructs were not interpreted using alpha-if-item-deleted because deleting one item leaves only one item, making Cronbach's alpha not meaningful.